# 🎤 AI Interview Coach — Powered by Google Gemma 4

Practice your interviews with a **human-sounding AI interviewer** that listens and responds with natural voice.

### ✨ Features
- **4 Interview Modes**: Behavioral, Case (Consulting), Technical, and General Mock
- **Natural Human Voice**: Powered by Edge TTS with multiple voice options
- **Voice Input**: Speak your answers using your microphone
- **Text Input**: Type your answers if you prefer
- **Real-time Transcript**: Full conversation history with timestamps
- **Performance Feedback**: Get a detailed scorecard at the end

### 🔑 Prerequisites
1. Get a free **Google AI API Key** from [Google AI Studio](https://aistudio.google.com/apikey)
2. Store it in Colab Secrets (🔑 icon in left sidebar) as `GOOGLE_API_KEY`
   - Or you'll be prompted to enter it manually

---

In [ ]:
# ============================================================
# 📦 Step 1: Install Dependencies
# ============================================================

!pip install -q google-genai edge-tts gradio SpeechRecognition pydub
!apt-get install -qq ffmpeg > /dev/null 2>&1

print("\n✅ All dependencies installed successfully!")

## ⚙️ Configuration & Setup

This cell sets up the API key, imports, and configures the system prompts for each interview type.

> **Tip**: Store your API key in Colab Secrets (🔑 sidebar) as `GOOGLE_API_KEY` for seamless access.

In [ ]:
# ============================================================
# ⚙️ Step 2: Imports & API Configuration
# ============================================================

import os
import tempfile
import asyncio
import time
import threading
import warnings
warnings.filterwarnings('ignore')

from google import genai
from google.genai import types
import edge_tts
import speech_recognition as sr
import gradio as gr

# --- API Key Setup ---
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
    print("🔑 API key loaded from Colab Secrets.")
except Exception:
    pass

if not API_KEY:
    API_KEY = input("🔑 Enter your Google AI API Key: ").strip()

client = genai.Client(api_key=API_KEY)
print("✅ Google GenAI client initialized.")

# --- Model Configuration ---
# Available Gemma 4 models on Google AI API:
#   gemma-4-12b-it       — 12B dense, lightweight
#   gemma-4-26b-a4b-it   — 26B MoE (only 4B active), fast & efficient
#   gemma-4-31b-it       — 31B dense, strongest reasoning
#
# Change the line below to switch models:
MODEL_NAME = "gemma-4-12b-it"

print(f"🤖 Model: {MODEL_NAME}")

# Quick verification that the model is reachable
try:
    _test = client.models.generate_content(model=MODEL_NAME, contents="Say OK")
    print(f"✅ Model '{MODEL_NAME}' is reachable!")
except Exception as e:
    print(f"⚠️  Model '{MODEL_NAME}' failed: {e}")
    print("\n🔄 Trying fallback models...")
    for fallback in ["gemma-4-26b-a4b-it", "gemma-4-31b-it", "gemma-4-12b-it", "gemma-3-27b-it"]:
        if fallback == MODEL_NAME:
            continue
        try:
            _test = client.models.generate_content(model=fallback, contents="Say OK")
            MODEL_NAME = fallback
            print(f"✅ Switched to fallback model: {MODEL_NAME}")
            break
        except Exception:
            print(f"   ❌ {fallback} — not available")
    else:
        print("\n❌ No Gemma 4 model found. Listing all available models:")
        for m in client.models.list():
            if 'gemma' in m.name.lower():
                print(f"   • {m.name}")

In [ ]:
# ============================================================
# 🎭 Step 3: Interview System Prompts
# ============================================================

SYSTEM_PROMPTS = {
    "🤝 Behavioral Interview": """You are a senior HR interviewer at a Fortune 500 company conducting a behavioral interview.

RULES:
- Start by warmly greeting the candidate and asking them to introduce themselves briefly.
- Ask ONE behavioral question at a time using the STAR method (Situation, Task, Action, Result).
- After each answer, give brief constructive feedback (2-3 sentences) highlighting what was good and what could be improved.
- Then ask a natural follow-up question OR move to a new behavioral question.
- Cover these areas across the interview: Leadership, Teamwork, Conflict Resolution, Problem-Solving, Adaptability, Communication, Time Management.
- Be encouraging but honest. Push the candidate to be specific and give concrete examples.
- If the candidate gives a vague answer, probe deeper: "Can you give me a specific example?" or "Walk me through exactly what you did."
- Keep your responses conversational, concise, and natural — you are speaking out loud, not writing an essay.
- Use a professional but warm tone, as if you genuinely want the candidate to succeed.
- Do NOT list multiple questions at once. One question at a time, like a real interview.""",

    "💼 Case Interview (Consulting)": """You are a senior partner at McKinsey & Company conducting a case interview.

RULES:
- Start by briefly introducing yourself and the case format.
- Present ONE well-structured business case scenario (e.g., market entry, profitability decline, M&A evaluation, pricing strategy, or market sizing).
- Give the case context in 3-4 sentences, like a real case interview.
- Let the candidate drive the case — wait for them to ask clarifying questions and propose a framework.
- Provide data ONLY when the candidate asks the right questions. Don't volunteer information.
- Evaluate their structured thinking: Do they break down the problem logically? Do they use appropriate frameworks?
- If they go off track, give subtle hints: "That's interesting, but what might be another angle to consider?"
- Test mental math by providing numbers when appropriate. Don't simplify — give realistic figures.
- After they propose a recommendation, challenge it: "The CEO asks why she should trust your recommendation. What would you say?"
- Keep responses concise and conversational — this is spoken dialogue, not written text.
- Be rigorous but supportive, like a real McKinsey interviewer who wants to see the candidate's best thinking.""",

    "💻 Technical Interview": """You are a senior software engineer at a top tech company conducting a technical interview.

RULES:
- Start with a brief introduction and ask the candidate about their technical background.
- Ask ONE technical question at a time, progressing from easier to harder.
- Mix question types: system design, algorithms & data structures, coding concepts, architecture decisions.
- For coding questions, discuss the approach verbally — don't ask them to write full code since this is voice-based.
- Evaluate problem-solving approach, not just correct answers. Ask "How would you approach this?" and "What tradeoffs are you considering?"
- If they're stuck, provide hints incrementally rather than giving the answer.
- Ask follow-up questions: "How would this scale?" "What if we had 10x the traffic?" "What's the time complexity?"
- Be conversational and encouraging. This is a dialogue, not an interrogation.
- Keep responses concise — speak naturally as if in a real interview room.""",

    "🎯 General Mock Interview": """You are an experienced professional interview coach conducting a comprehensive mock interview.

RULES:
- Start with a warm greeting and ask what role/industry the candidate is preparing for.
- Adapt your questions to their target role and experience level.
- Mix different question types: tell-me-about-yourself, behavioral, situational, motivational, and role-specific.
- Ask common but important questions like: "Why this role?", "Where do you see yourself in 5 years?", "What's your biggest weakness?"
- After each answer, provide specific coaching feedback: what was strong, what to improve, and a quick tip.
- Be like a supportive coach — direct and honest, but always constructive.
- Keep responses natural and conversational. One question at a time.
- Help them improve their delivery, not just content. Comment on structure, specificity, and confidence."""
}

# --- Voice Options (Edge TTS - Natural Human Voices) ---
VOICES = {
    "👩 Aria (US Female)": "en-US-AriaNeural",
    "👨 Guy (US Male)": "en-US-GuyNeural",
    "👩 Jenny (US Female)": "en-US-JennyNeural",
    "👩 Sonia (UK Female)": "en-GB-SoniaNeural",
    "👨 Ryan (UK Male)": "en-GB-RyanNeural",
    "👩 Neerja (Indian Female)": "en-IN-NeerjaNeural",
    "👨 Prabhat (Indian Male)": "en-IN-PrabhatNeural"
}

print("✅ System prompts and voice options configured.")
print(f"📋 Interview modes: {', '.join(SYSTEM_PROMPTS.keys())}")
print(f"🗣️ Voice options: {', '.join(VOICES.keys())}")

## 🔊 Audio & Interview Engine

This cell contains the core logic for:
- **Text-to-Speech (TTS)**: Converts AI responses to natural human voice using Edge TTS
- **Speech-to-Text (STT)**: Converts your voice recordings to text using Google Speech Recognition
- **Interview Logic**: Manages conversation flow, history, and feedback generation

In [ ]:
# ============================================================
# 🔊 Step 4: Audio Functions (TTS & STT)
# ============================================================

async def text_to_speech(text, voice_key):
    """Convert text to natural-sounding speech using Edge TTS."""
    voice_id = VOICES.get(voice_key, "en-US-AriaNeural")
    output_file = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    output_path = output_file.name
    output_file.close()

    try:
        communicate = edge_tts.Communicate(text, voice_id, rate="+5%")
        await communicate.save(output_path)
        return output_path
    except Exception as e:
        print(f"❌ TTS Error: {e}")
        return None


def tts_sync(text, voice_key):
    """Synchronous wrapper — runs TTS in a fresh event loop on a dedicated thread."""
    result = [None]
    def _run():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result[0] = loop.run_until_complete(text_to_speech(text, voice_key))
        finally:
            loop.close()
    t = threading.Thread(target=_run)
    t.start()
    t.join()
    return result[0]


def speech_to_text(audio_path):
    """Convert audio file to text using Google Speech Recognition."""
    if audio_path is None:
        return None

    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 300

    try:
        with sr.AudioFile(audio_path) as source:
            audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "[Could not understand the audio. Please try again more clearly.]"
    except sr.RequestError as e:
        return f"[Speech recognition service error: {e}]"
    except Exception as e:
        return f"[Audio processing error: {e}]"


print("✅ Audio functions ready.")


# ============================================================
# 🧠 Step 5: Interview Session Manager
# ============================================================

class InterviewSession:
    """Manages the state of an interview session."""

    def __init__(self):
        self.chat = None
        self.mode = None
        self.is_active = False
        self.question_count = 0
        self.start_time = None

    def start(self, interview_type):
        """Start a new interview session."""
        self.mode = interview_type
        self.is_active = True
        self.question_count = 0
        self.start_time = time.time()

        system_prompt = SYSTEM_PROMPTS[interview_type]

        self.chat = client.chats.create(
            model=MODEL_NAME,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.8,
                max_output_tokens=600,
                top_p=0.95,
                top_k=40,
            )
        )

    def send_message(self, user_message):
        """Send a message and get a response."""
        if not self.is_active or self.chat is None:
            return "Please start an interview first."

        self.question_count += 1
        response = self.chat.send_message(user_message)
        return response.text

    def get_feedback(self):
        """Request end-of-interview feedback."""
        if not self.is_active or self.chat is None:
            return "No active interview to evaluate."

        elapsed = int(time.time() - self.start_time) if self.start_time else 0
        minutes = elapsed // 60

        feedback_prompt = (
            "The interview is now over. We had a conversation for about "
            f"{minutes} minutes with {self.question_count} exchanges. "
            "Please provide a comprehensive performance review with: "
            "1) Overall rating out of 10, "
            "2) Top 3 strengths demonstrated, "
            "3) Top 3 areas for improvement, "
            "4) Specific actionable tips for their next interview. "
            "Be honest, detailed, and constructive."
        )

        response = self.chat.send_message(feedback_prompt)
        self.is_active = False
        return response.text

    def reset(self):
        """Reset the session."""
        self.chat = None
        self.mode = None
        self.is_active = False
        self.question_count = 0
        self.start_time = None


# Global session instance
session = InterviewSession()

print("✅ Interview engine ready.")

## 🚀 Launch the Interview Coach

Run the cell below to launch the interactive interview interface.

### How to Use
1. **Select** your interview type and preferred interviewer voice
2. Click **🚀 Start Interview** — the AI will greet you and ask the first question
3. **Record** your answer using the microphone OR **type** it in the text box
4. Click **🎤 Send Voice** or **📤 Send Text** to submit
5. The AI will respond with voice and text — listen and respond naturally
6. When done, click **📝 End & Get Feedback** for a detailed performance review

> **Note**: The public share link lets you access the app from any device, including your phone!

In [ ]:
# ============================================================
# 🚀 Step 6: Gradio Interactive Interface
# ============================================================

def start_interview(interview_type, voice):
    """Initialize a new interview session."""
    try:
        session.reset()
        session.start(interview_type)

        # Get the opening message from the AI
        opening = session.send_message(
            "Start the interview now. Greet the candidate warmly and ask your first question."
        )

        # Convert to speech
        audio_path = tts_sync(opening, voice)

        # Build initial chat history
        history = []
        history.append({"role": "assistant", "content": f"🎤 *{interview_type} started*\n\n{opening}"})

        status = f"✅ {interview_type} in progress | Voice: {voice}"
        return history, audio_path, status

    except Exception as e:
        error_msg = f"❌ Error starting interview: {str(e)}"
        return [{"role": "assistant", "content": error_msg}], None, error_msg


def process_voice_input(audio, voice, history):
    """Process voice input from the microphone."""
    if not session.is_active:
        return history or [], None, "⚠️ Please start an interview first!"

    if audio is None:
        return history or [], None, "⚠️ No audio detected. Please try recording again."

    # Convert speech to text
    user_text = speech_to_text(audio)

    if user_text is None or user_text.startswith("["):
        return history or [], None, f"⚠️ {user_text}"

    return _handle_response(user_text, voice, history)


def process_text_input(text_input, voice, history):
    """Process typed text input."""
    if not session.is_active:
        return history or [], None, "⚠️ Please start an interview first!", ""

    if not text_input or text_input.strip() == "":
        return history or [], None, "⚠️ Please type your response.", text_input

    result = _handle_response(text_input.strip(), voice, history)
    return result[0], result[1], result[2], ""  # Clear the text input


def _handle_response(user_text, voice, history):
    """Common handler for both voice and text input."""
    history = history or []

    # Add user message to history
    history.append({"role": "user", "content": user_text})

    # Get AI response
    try:
        ai_response = session.send_message(user_text)
    except Exception as e:
        error = f"❌ Model error: {str(e)}"
        history.append({"role": "assistant", "content": error})
        return history, None, error

    # Convert response to speech
    audio_path = tts_sync(ai_response, voice)

    # Add AI response to history
    history.append({"role": "assistant", "content": ai_response})

    status = f"💬 Exchange #{session.question_count} | {session.mode}"
    return history, audio_path, status


def end_interview(voice, history):
    """End the interview and get performance feedback."""
    if not session.is_active:
        return history or [], None, "⚠️ No active interview to end."

    history = history or []
    history.append({"role": "user", "content": "*[Interview ended — requesting feedback]*"})

    try:
        feedback = session.get_feedback()
    except Exception as e:
        error = f"❌ Error getting feedback: {str(e)}"
        history.append({"role": "assistant", "content": error})
        return history, None, error

    audio_path = tts_sync(feedback, voice)

    history.append({"role": "assistant", "content": f"📝 **Performance Review**\n\n{feedback}"})

    status = "📝 Interview complete! Review your feedback above."
    return history, audio_path, status


# --- Build the Gradio Interface ---

CUSTOM_CSS = """
.gradio-container {
    max-width: 900px !important;
    margin: auto !important;
}
footer { display: none !important; }
#title {
    text-align: center;
    font-size: 2em;
    font-weight: 700;
    margin-bottom: 0.2em;
}
#subtitle {
    text-align: center;
    font-size: 1.1em;
    color: #888;
    margin-bottom: 1.5em;
}
#status-bar {
    font-size: 0.95em;
    padding: 8px 12px;
    border-radius: 8px;
}
"""

with gr.Blocks(
    title="AI Interview Coach — Gemma 4",
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    ),
    css=CUSTOM_CSS,
) as demo:

    # --- Header ---
    gr.Markdown("# 🎤 AI Interview Coach", elem_id="title")
    gr.Markdown(
        "Practice behavioral, case & technical interviews with a human-sounding AI — powered by **Gemma 4**",
        elem_id="subtitle"
    )

    # --- Status Bar ---
    status_bar = gr.Textbox(
        value="👋 Select an interview type and click Start!",
        label="Status",
        interactive=False,
        elem_id="status-bar",
    )

    # --- Config Row ---
    with gr.Row():
        interview_type = gr.Dropdown(
            choices=list(SYSTEM_PROMPTS.keys()),
            value="🤝 Behavioral Interview",
            label="📋 Interview Type",
            scale=2,
        )
        voice_select = gr.Dropdown(
            choices=list(VOICES.keys()),
            value="👩 Aria (US Female)",
            label="🗣️ Interviewer Voice",
            scale=2,
        )

    # --- Control Buttons ---
    with gr.Row():
        start_btn = gr.Button("🚀 Start Interview", variant="primary", scale=2)
        end_btn = gr.Button("📝 End & Get Feedback", variant="stop", scale=2)

    # --- Chat History ---
    chatbot = gr.Chatbot(
        label="💬 Interview Conversation",
        height=450,
        type="messages",
        show_copy_button=True,
        avatar_images=(
            None,  # User avatar
            "https://fonts.gstatic.com/s/i/productlogos/gemini/v8/web-24dp/logo_gemini_color_1x_web_24dp.png"  # AI avatar
        ),
    )

    # --- AI Voice Response ---
    ai_audio = gr.Audio(
        label="🔊 Interviewer's Voice",
        type="filepath",
        autoplay=True,
        interactive=False,
    )

    # --- Input Section ---
    gr.Markdown("### 💬 Your Response")

    with gr.Tab("🎤 Voice Input"):
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Record your answer",
        )
        voice_submit_btn = gr.Button("🎤 Send Voice Response", variant="primary")

    with gr.Tab("⌨️ Text Input"):
        text_input = gr.Textbox(
            placeholder="Type your answer here...",
            label="Your Answer",
            lines=3,
        )
        text_submit_btn = gr.Button("📤 Send Text Response", variant="primary")

    # --- Event Handlers ---
    start_btn.click(
        fn=start_interview,
        inputs=[interview_type, voice_select],
        outputs=[chatbot, ai_audio, status_bar],
    )

    voice_submit_btn.click(
        fn=process_voice_input,
        inputs=[audio_input, voice_select, chatbot],
        outputs=[chatbot, ai_audio, status_bar],
    )

    text_submit_btn.click(
        fn=process_text_input,
        inputs=[text_input, voice_select, chatbot],
        outputs=[chatbot, ai_audio, status_bar, text_input],
    )

    end_btn.click(
        fn=end_interview,
        inputs=[voice_select, chatbot],
        outputs=[chatbot, ai_audio, status_bar],
    )

    # --- Footer ---
    gr.Markdown(
        "---\n"
        "<center>\n"
        "<small>Built with ❤️ using Google Gemma 4 · Edge TTS · Gradio</small>\n"
        "</center>"
    )


# --- Launch ---
print("\n" + "="*60)
print("🚀 Launching AI Interview Coach...")
print("="*60)
demo.launch(debug=True, share=True, quiet=False)